# Compare our output to the Thrasher implementation

## running this notebook

```Python
uv run coiled notebook start --vm-type m8g.16xlarge --region 'us-west-2' --tag Project=SRM --sync
```

This notebook compares downscaled data when running our approach trained on (1) ERA5 and (2) PGMF. We then take (2) and are able to compare it to the NEX-GDDP dataset which was trained on the same dataset. These two should be ~similar since we are largely just porting over the implementation into Python from NCL, and have implemented it largely the same.

First we do a baseline comparison between our implementation using the two different training datasets.

In [1]:
import matplotlib.pyplot as plt

from srm import catalog
from srm.utils import open_icechunk, resolve_s3_glob

# Goal
- regional run with the 007 ensemble member historical period and ssp245 to get pr (and tas/rsds) for comparison with nex-gddp
- this will be a different ensemble member from the snapshot so it has to be done separately
- we will do this once now. if there are other substantial changes to code base then we can update the comparison.

In [7]:
def get_fname(version="production", gcm="CESM2-WACCM", training_dataset="ERA5"):
    fname = (
        "s3://carbonplan-srm/output/"
        + version +
        "/" +
        gcm +
        "/" +
        training_dataset +
        "-global.icechunk/"
    )
    # s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk

    return fname

In [3]:
def plot_comparison(da1, da2, subtitle1="", subtitle2="", suptitle=""):
    diff = da1 - da2
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    da1.plot(ax=axes[0], cmap="viridis")
    axes[0].set_title(subtitle1)
    da2.plot(ax=axes[1], cmap="viridis")
    axes[1].set_title(subtitle2)
    diff.plot(ax=axes[2], cmap="RdBu_r", robust=True)
    axes[2].set_title(f"difference ({subtitle1} - {subtitle2})")
    fig.suptitle(suptitle)
    plt.tight_layout()

We'll keep the comparison to `tas` and `ssp245` for now. First read in the dataset we made training to PGMF.

In [21]:
import icechunk
import xarray as xr

ds_hist_coarse = xr.open_zarr(
    session.store,
    group="debiased_coarse/historical/tas/r1i1p1f1",
    consolidated=False,
    zarr_format=3,
    chunks="auto",
)

In [10]:
fname = get_fname(
    version="production", gcm="CESM2-WACCM", training_dataset="ERA5"
)
cp_pgmf = open_icechunk(path=fname)[var]
coarse = catalog.get("CESM2-WACCM-SSP245-icechunk").to_xarray()["tas"].sel(ensemble_member="003")

IcechunkError:   x the repository doesn't exist
  | 
  | context:
  |    0: icechunk::repository::open
  |              at icechunk/src/repository.rs:344
  | 
  `-> the repository doesn't exist


Then we do the apples-to-apples comparison between nex-gddp and our PGMF implementation.

In [ ]:
# read in the nex-gddp dataset (Thrasher et al (2022)'s implementation, trained on PGMF)

In [ ]:
nex = catalog.get("NASA-NEX-SSP245").to_xarray()
nex["time"] = nex.indexes["time"].to_datetimeindex()
# subset to the area that this test area covers
nex = nex.sel(lat=cp_pgmf.lat, lon=cp_pgmf.lon, time=slice("2015", "2099"))[var].load()

In [ ]:
cp_pgmf = cp_pgmf.sel(
    time=slice("2015", "2099")
).load()  # our dataset includes 2100 but nex does not so subset to same time range

# compare a pi day for of nex-gddp and our PGMF implementation

In [ ]:
single_day = "2015-03-14"
plot_comparison(
    cp_pgmf.sel(time=single_day).load(),
    nex.sel(time=single_day).load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

In [ ]:
single_day = "2050-12-14"
plot_comparison(
    cp_pgmf.sel(time=single_day).load(),
    nex.sel(time=single_day).load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

In [ ]:
single_day = "2080-08-14"
plot_comparison(
    cp_pgmf.sel(time=single_day).load(),
    nex.sel(time=single_day).load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

# compare mean tas

In [ ]:
plot_comparison(
    cp_pgmf.mean(dim="time"), nex.mean(dim="time"), subtitle1="CP", subtitle2="NEX-GDDP"
)

# compare 99p tas

Over the south africa example the warmest tempeartures (1p) are generally low biased in CP by ~1 degree.

In [ ]:
plot_comparison(
    cp_pgmf.quantile(q=0.99, dim="time").load(),
    nex.quantile(q=0.99, dim="time").load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

# compare 01p tas

Over the south africa example the coolest tempeartures (1p) are generally high biased in CP by up to ~0.5 to 1 degree.

In [ ]:
plot_comparison(
    cp_pgmf.quantile(q=0.01, dim="time").load(),
    nex.quantile(q=0.01, dim="time").load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

Look at individual timeseries of a year of actual data to confirm same ensemble member and look for patterns of deviations. In the plots below we see that the timeseries on individual days can deviate by ~1 degree but largely track.

In [ ]:
lat = -30.125
lon = 20.125
cp_pgmf.sel(lat=lat, lon=lon, time=slice("2041", "2041")).plot(label="CP-PGMF")
nex.sel(lat=lat, lon=lon, time=slice("2041", "2041")).plot(label="NEX-GDDP")
coarse.sel(lat=lat, lon=lon, method="nearest").sel(time=slice("2041", "2041")).plot(
    label="Coarse GCM"
)
plt.legend()

In [ ]:
lat = -30.125
lon = 20.125
cp_pgmf.sel(lat=lat, lon=lon, time=slice("2081", "2081")).plot(label="CP-PGMF")
nex.sel(lat=lat, lon=lon, time=slice("2081", "2081")).plot(label="NEX-GDDP")
coarse.sel(lat=lat, lon=lon, method="nearest").sel(time=slice("2081", "2081")).plot(
    label="Coarse GCM"
)

plt.legend()

Let's inspect the days when our approach is markedly cooler than NEX ~August 5 2081.

In [ ]:
single_day = "2081-08-05"
plot_comparison(
    cp_pgmf.sel(time=single_day).load(),
    nex.sel(time=single_day).load(),
    subtitle1="CP",
    subtitle2="NEX-GDDP",
)

Check to see if there are any patterns in the RMSE. Below the RMSE has grid artifacts similar to the raw GCM, so it suggests that something in the disaggregation step is contributing to any differences between the steps.

In [ ]:
import numpy as np

nex_sq = nex.squeeze("ensemble_member")
rmse = np.sqrt(((cp_pgmf - nex_sq) ** 2).mean(dim="time")).load()

fig, ax = plt.subplots()
rmse.plot(ax=ax, vmax=1, vmin=0)
ax.set_title("RMSE: CP-PGMF vs NEX-GDDP (all time)")
plt.tight_layout()